# 07 · Tính tay theo Word Final v7.1
Các số hai chiều là **ví dụ giả định trong Chương 2**, không phải hidden thật của pretrained model. Mỗi cell thực hiện một bước. SGD một nhãn ở dưới khác AdamW và BCE trung bình 28 nhãn dùng khi train.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/notebook_support.py').is_file())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
EXPERIMENT = ROOT/'experiments/best_epochs_20261103'
MAIN = EXPERIMENT/'main_results'
ERRORS = ROOT/'experiments/threshold_error_20261110'
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 40)
pd.set_option('display.float_format', lambda x: f'{x:.6f}')
print('Báo cáo Final v7.1 · chỉ đánh giá từ dữ liệu/điểm đã lưu, không huấn luyện lại.')
from scipy.special import expit,softmax
from scipy.special import erf
pd.set_option('display.precision',9)
print('Tất cả phép tính dưới dùng float64 trên CPU.')
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda x: f'{x:.9f}')

Báo cáo Final v7.1 · chỉ đánh giá từ dữ liệu/điểm đã lưu, không huấn luyện lại.


Tất cả phép tính dưới dùng float64 trên CPU.


## 1. TF-IDF — từ văn bản đến vector
TF là số lần từ xuất hiện trong câu; DF là số câu chứa từ; IDF = ln((N+1)/(DF+1))+1. Nhân TF×IDF rồi chuẩn hóa L2. Đối chiếu Word 2.2.1 và PPT slide 9.

In [2]:
from src.learning.baseline_numpy import simple_tfidf
X,words,idf=simple_tfidf(['thank happy','thank','sad'])
assert words==['happy','sad','thank']
display(pd.DataFrame({'từ':words,'IDF':idf}))
display(pd.DataFrame(X,columns=words,index=['D1','D2','D3']))
w_demo=np.array([.2,-1.,2.]);bias_demo=-.5
z_tfidf=float(X[0]@w_demo+bias_demo)
print('D1: logit =',z_tfidf,'; sigmoid =',expit(z_tfidf))
assert round(z_tfidf,7)==.8698891 and round(float(expit(z_tfidf)),7)==.7047226
print('PASS: trọng số [0,2;−1;2], bias −0,5 và đầu ra khớp Word 2.2.1 / PPT slide 9.')

,từ,IDF
0,happy,1.693147181
1,sad,1.693147181
2,thank,1.287682072


,happy,sad,thank
D1,0.795960542,0.000000000,0.605348508
D2,0.000000000,0.000000000,1.000000000
D3,0.000000000,1.000000000,0.000000000


D1: logit = 0.8698891245262161 ; sigmoid = 0.7047226265945739
PASS: trọng số [0,2;−1;2], bias −0,5 và đầu ra khớp Word 2.2.1 / PPT slide 9.


## 2. Self-attention — tổng có trọng số của hai value
Q=K=V=I₂; S=QKᵀ/√2; A=softmax(S) theo hàng; O=AV. Softmax ở đây chạy theo vị trí token, không phải theo nhãn cảm xúc.

In [3]:
X=np.eye(2);Q=K=V=X
S=Q@K.T/np.sqrt(2);A=softmax(S,axis=1);O=A@V
for name,value in [('Q=K=V',X),('S',S),('A',A),('O',O)]:
    print(name);display(pd.DataFrame(value))
np.testing.assert_allclose(O[0],[.6697615493,.3302384507],atol=1e-9)
print('Mô hình thật: d_k=768/12=64, nên chia sqrt(64)=8.')

Q=K=V


,0,1
0,1.000000000,0.000000000
1,0.000000000,1.000000000


S


,0,1
0,0.707106781,0.000000000
1,0.000000000,0.707106781


A


,0,1
0,0.669761549,0.330238451
1,0.330238451,0.669761549


O


,0,1
0,0.669761549,0.330238451
1,0.330238451,0.669761549


Mô hình thật: d_k=768/12=64, nên chia sqrt(64)=8.


## 3. Residual → LayerNorm → FFN/GELU
Residual cộng lại đầu vào. LayerNorm chuẩn hóa trên chiều đặc trưng của từng token. FFN đổi biểu diễn của từng token; ví dụ dùng GELU theo x·Φ(x).

In [4]:
residual=X[0]+O[0]
eps=1e-12
normalized=(residual-residual.mean())/np.sqrt(residual.var()+eps)
W1=np.array([[1.,0.],[0.,1.],[-1.,0.],[0.,-1.]])
W2=np.array([[1.,0.,0.,0.],[0.,1.,0.,0.]])
hidden=W1@normalized
gelu=lambda x:.5*x*(1+erf(x/np.sqrt(2)))
display(pd.DataFrame({'bước':['Residual','LayerNorm (epsilon=1e-12)','FFN: nhân W1','GELU'],
                      'vector':[residual.tolist(),normalized.tolist(),hidden.tolist(),gelu(hidden).tolist()]}))
print('Bản tính tay làm tròn LayerNorm thành [1,-1] trước FFN:')
illustrative=W1@np.array([1.,-1.]);ffn=W2@gelu(illustrative)
display(pd.Series({'W1 đầu ra':illustrative.tolist(),'GELU':gelu(illustrative).tolist(),
                  'FFN sau W2':ffn.tolist(),'Residual thứ hai':(np.array([1.,-1.])+ffn).tolist()}))

,bước,vector
0,Residual,"[1.6697615493266569, 0.3302384506733431]"
1,LayerNorm (epsilon=1e-12),"[0.9999999999988854, -0.9999999999988854]"
2,FFN: nhân W1,"[0.9999999999988854, -0.9999999999988854, -0.9999999999988854, 0.9999999999988854]"
3,GELU,"[0.8413447460673356, -0.15865525393154997, -0.15865525393154997, 0.8413447460673356]"


Bản tính tay làm tròn LayerNorm thành [1,-1] trước FFN:


W1 đầu ra                                                                         [1.0, -1.0, -1.0, 1.0]
GELU                [0.8413447460685429, -0.15865525393145707, -0.15865525393145707, 0.8413447460685429]
FFN sau W2                                                    [0.8413447460685429, -0.15865525393145707]
Residual thứ hai                                               [1.8413447460685428, -1.1586552539314572]
dtype: object

## 4. BERT: hidden [CLS] → pooler → classifier
Chọn h=atanh([0,6;−0,2]), Wp=I, bp=0 để tanh trả đúng u=[0,6;−0,2]. Đây là cách dựng số minh họa; h đã là hidden sau encoder.

In [5]:
h=np.arctanh(np.array([.6,-.2]));u=np.tanh(h)
w=np.array([1.2,-.4]);bias=.1
z=float(w@u+bias);p=float(expit(z))
display(pd.Series({'hidden h':h.tolist(),'pooler u':u.tolist(),'logit z':z,'sigmoid p':p}))
assert abs(p-.7109495026250039)<1e-12

hidden h     [0.6931471805599453, -0.2027325540540822]
pooler u                   [0.6, -0.19999999999999998]
logit z                                    0.900000000
sigmoid p                                  0.710949503
dtype: object

## 5. Hai head còn lại — so theo đúng ví dụ của báo cáo
RoBERTa: dropout → dense → tanh → dropout → out_proj. DistilBERT: pre_classifier → ReLU → classifier. Dropout tắt khi minh họa.

In [6]:
h_roberta=np.array([1.,-1.]);dense=.5*h_roberta
u_roberta=np.tanh(dense);z_roberta=float(np.array([1.,-1.])@u_roberta)
h_distil=np.array([.6,-.2]);u_distil=np.maximum(h_distil,0)
z_distil=float(w@u_distil+bias)
heads=pd.DataFrame([{'mô hình':'BERT','logit':z,'sigmoid':p},
                    {'mô hình':'RoBERTa','logit':z_roberta,'sigmoid':float(expit(z_roberta))},
                    {'mô hình':'DistilBERT','logit':z_distil,'sigmoid':float(expit(z_distil))}])
display(heads)
print('Ba xác suất là số minh họa, không dùng xếp hạng mô hình.')

,mô hình,logit,sigmoid
0,BERT,0.900000000,0.710949503
1,RoBERTa,0.924234315,0.715904090
2,DistilBERT,0.820000000,0.694236340


Ba xác suất là số minh họa, không dùng xếp hạng mô hình.


## 6. BCE một nhãn → gradient → SGD
Với y=1: loss=−ln(p), ∂loss/∂z=p−y, ∂loss/∂w=(p−y)u. SGD: w_mới=w−η·gradient. Đây là phép tính một nhãn, **không chia thêm cho 2**. Đối chiếu Word 2.3.9 và PPT slide 17.

In [7]:
y=1.;eta=2e-5
loss=float(np.logaddexp(0,z)-y*z);grad_z=p-y;grad_w=grad_z*u
w_new=w-eta*grad_w
display(pd.Series({'BCE một nhãn':loss,'gradient logit':grad_z,'gradient w1':grad_w[0],
                   'w1 mới':w_new[0],'w1 mới làm tròn 7 chữ số':round(w_new[0],7)}))
eps_diff=1e-5
objective=lambda weight:np.logaddexp(0,weight@u+bias)-y*(weight@u+bias)
numerical=(objective(w+np.array([eps_diff,0]))-objective(w-np.array([eps_diff,0])))/(2*eps_diff)
np.testing.assert_allclose(numerical,grad_w[0],atol=1e-9)
assert round(w_new[0],7)==1.2000035
print('PASS: gradient giải tích khớp sai phân hữu hạn; SGD khớp báo cáo.')

BCE một nhãn                0.341153875
gradient logit             -0.289050497
gradient w1                -0.173430298
w1 mới                      1.200003469
w1 mới làm tròn 7 chữ số    1.200003500
dtype: float64

PASS: gradient giải tích khớp sai phân hữu hạn; SGD khớp báo cáo.


## 7. Chưng cất: KL từ hai phân phối giả định
KL(p||q)=Σp·ln(p/q). Đây là ví dụ giải thích pretrained DistilBERT, không phải loss fine-tune C3 trong đồ án.

In [8]:
temperature=2.
teacher=softmax(np.array([2.,1.,0.])/temperature)
student=softmax(np.array([1.5,1.,.5])/temperature)
kl=float(np.sum(teacher*np.log(teacher/student)))
display(pd.Series({'teacher q':teacher.tolist(),'student p':student.tolist(),'KL(q || p)':kl,'T² × KL':temperature**2*kl}))
assert round(kl,4)==.0191
print('PASS: ví dụ nhiệt độ T=2 khớp KL 0,0191 trong Word.')

teacher q        [0.506480391055654, 0.3071958857184984, 0.1863237232258476]
student p     [0.41922895160969764, 0.32649583579983665, 0.2542752125904656]
KL(q || p)                                                       0.019107581
T² × KL                                                          0.076430323
dtype: object

PASS: ví dụ nhiệt độ T=2 khớp KL 0,0191 trong Word.


## 8. Cấu hình đã chạy: đủ ba seed, không huấn luyện lại

In [9]:
lock=json.loads((MAIN/'LOCK_BEFORE_TEST.json').read_text(encoding='utf-8'))
rows=[]
for run in lock['runs']:
    meta=json.loads((MAIN/f"training_receipts/{run['architecture']}/seed_{run['seed']}/run_metadata.json").read_text(encoding='utf-8'))
    rows.append({'mô hình':run['architecture'],'seed':run['seed'],'epoch chọn':run['selected_epoch'],**{k:meta['config'][k] for k in ['learning_rate','epochs','batch_size','max_length']}})
display(pd.DataFrame(rows))
print('Checkpoint chọn theo Macro-F1 validation @0,5; không chọn lại theo test.')

,mô hình,seed,epoch chọn,learning_rate,epochs,batch_size,max_length
0,bert,42,7,0.000020000,8,16,128
1,bert,123,6,0.000020000,8,16,128
2,bert,2026,7,0.000020000,8,16,128
3,roberta,42,7,0.000020000,8,16,128
4,roberta,123,8,0.000020000,8,16,128
5,roberta,2026,5,0.000020000,8,16,128
6,distilbert,42,7,0.000020000,8,16,128
7,distilbert,123,8,0.000020000,8,16,128
8,distilbert,2026,8,0.000020000,8,16,128


Checkpoint chọn theo Macro-F1 validation @0,5; không chọn lại theo test.
